In [1]:
import sys
import os
import re
import json
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from tqdm import tqdm

PROJECT_ROOT = Path("../../").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from lib.llm.llm_client import MODEL, MODEL_N, COSTS, timed_completion, strip_thinking

In [2]:
df = pd.read_csv("../../datasets/cfb_perturbed.csv")


In [3]:
N = None   # set to 100 for a test run
df_run = df.head(N) if N else df

df.head()


,Company's name,Fiscal year,Question ID,Question,Type of question,Answer,Documents,Pages,Document extracts,Extract type,perturbation_type,perturbation_level
0,Ali Baba Group,2024,Q1,According to the company's Disclosure from FY2...,LR,According to the company's statement for fisca...,"2024 Alibaba Group Environmental, Social and G...",doc1{page7},doc1{\nRestoring Our Green Planet\nAddressing ...,text,none,0
1,Ali Baba Group,2024,Q2,What drove carbon intensity change as of the F...,PE,Ali Baba Group carbon intensity dropped in FY2...,"2024 Alibaba Group Environmental, Social and G...","doc1{page10, page11}",doc1{Emissions reduction from our own operatio...,text,none,0
2,Ali Baba Group,2024,Q3,Does the company have a decarbonization trajec...,LR,"\nNo, the company has several projects to deca...","2024 Alibaba Group Environmental, Social and G...","doc1{page31, page37, page157 }",doc1{We dug deep into four major scenarios for...,text,none,0
3,Ali Baba Group,2024,Q4,Does the company operate in a high emitting se...,LR,"Yes, Alibaba Group operates in the logistics s...","2024 Alibaba Group Environmental, Social and G...",doc1{page29},doc1{Reducing emissions in the logistics secto...,text,none,0
4,Ali Baba Group,2024,Q6,Has the company identified significant decarbo...,PE,"Yes, Alibaba Group has identified several sign...","2024 Alibaba Group Environmental, Social and G...","doc1{page26, page31, page33, page34}",doc1{Improving efficiency through technologies...,text,none,0


In [4]:

ANSWER_PROMPT_LR = """
You are answering a complex financial question that requires multi-step reasoning.
Use only the document extracts provided.

Question:
{question}

Document extracts:
{extracts}

Instructions:
- Think through the problem step by step.
- Show your reasoning chain explicitly before stating the final answer.
- Label each reasoning step clearly (Step 1, Step 2, ...).
- Conclude with a clearly marked Final Answer.
- Do not hallucinate numbers or facts not present in the extracts.
"""

ANSWER_PROMPT_PE = """
You are extracting a specific piece of information from financial documents.
Use only the document extracts provided.

Question:
{question}

Document extracts:
{extracts}

Instructions:
- Locate the exact value, name, date, or fact being asked for.
- State the answer directly and concisely.
- Do not add explanation unless the question explicitly asks for it.
- If the answer is a number, include units and any relevant qualifiers (e.g. fiscal year, currency).
- Do not mention that you are using extracts.
"""

ANSWER_PROMPT_NR = """
You are solving a numerical reasoning problem using financial document extracts.
Use only the document extracts provided.

Question:
{question}

Document extracts:
{extracts}

Instructions:
- Identify all relevant numbers from the extracts.
- Show every arithmetic step explicitly.
- State the formula or operation used at each step.
- Conclude with a clearly marked Final Answer including units.
- Do not round intermediate values unless instructed.
- Do not hallucinate numbers not present in the extracts.
"""


JUDGE_PROMPT_LR = """
You are an expert evaluator judging a model answer to a Long Reasoning (LR) question.

LR questions require multi-step logical reasoning over financial documents.
The ground truth represents the expected conclusion and reasoning chain.

Evaluate on these dimensions (weight in brackets):
1. [30%] Logical coherence — does the reasoning chain hold together?
2. [25%] Factual correctness — are facts and figures drawn correctly from the extracts?
3. [25%] Completeness — are all necessary reasoning steps present?
4. [10%] Faithfulness — no hallucinated evidence beyond the extracts?
5. [10%] Final answer correctness — does the conclusion match the ground truth?

Scoring guidance:
- 1.000 = correct conclusion, complete reasoning, no hallucinations
- 0.700–0.999 = correct conclusion, minor gaps or imprecision in reasoning
- 0.400–0.699 = partially correct reasoning, conclusion may be off
- 0.100–0.399 = reasoning mostly wrong but some valid steps
- 0.000–0.099 = completely incorrect or irrelevant

Return ONLY valid JSON:
{{
  "score": 0.000,
  "reason": "brief explanation covering each dimension"
}}

Question:
{question}

Document Extracts:
{extracts}

Ground Truth:
{ground_truth}

Model Answer:
{model_answer}
"""

JUDGE_PROMPT_PE = """
You are an expert evaluator judging a model answer to a Point Extraction (PE) question.

PE questions ask the model to extract a specific fact, value, or entity from financial documents.
Precision and exactness matter most. Verbose answers that bury the correct value are penalised.

Evaluate on these dimensions (weight in brackets):
1. [50%] Exact match — does the extracted value match the ground truth exactly (or near-exactly with acceptable formatting variation)?
2. [25%] Precision — is the answer free of irrelevant information that obscures the answer?
3. [15%] Faithfulness — is the answer supported by the extracts, not hallucinated?
4. [10%] Format correctness — are units, currency, dates formatted appropriately?

Scoring guidance:
- 1.000 = exact match, clean, faithful
- 0.700–0.999 = correct value with minor formatting/unit differences
- 0.400–0.699 = partially correct (wrong qualifier, wrong year, wrong unit)
- 0.100–0.399 = wrong value but correct entity type
- 0.000–0.099 = completely wrong or hallucinated

Return ONLY valid JSON:
{{
  "score": 0.000,
  "reason": "brief explanation covering each dimension"
}}

Question:
{question}

Document Extracts:
{extracts}

Ground Truth:
{ground_truth}

Model Answer:
{model_answer}
"""

JUDGE_PROMPT_NR = """
You are an expert evaluator judging a model answer to a Numerical Reasoning (NR) question.

NR questions require arithmetic or quantitative reasoning over financial figures.
Both the reasoning process and the final numerical result are evaluated.

Evaluate on these dimensions (weight in brackets):
1. [40%] Final answer correctness — does the computed result match the ground truth (within acceptable rounding tolerance)?
2. [30%] Calculation correctness — are all arithmetic steps correct and shown?
3. [20%] Use of correct inputs — did the model use the right numbers from the extracts?
4. [10%] Faithfulness — no numbers hallucinated beyond what is in the extracts?

Scoring guidance:
- 1.000 = correct result, correct steps, correct inputs, no hallucinations
- 0.700–0.999 = correct result with minor rounding or presentation issues
- 0.400–0.699 = correct method but arithmetic error, or wrong input leading to wrong result
- 0.100–0.399 = wrong approach but some correct steps or values used
- 0.000–0.099 = completely incorrect

Return ONLY valid JSON:
{{
  "score": 0.000,
  "reason": "brief explanation covering each dimension"
}}

Question:
{question}

Document Extracts:
{extracts}

Ground Truth:
{ground_truth}

Model Answer:
{model_answer}
"""


ANSWER_PROMPT_BY_TYPE = {
    "LR": ANSWER_PROMPT_LR,
    "PE": ANSWER_PROMPT_PE,
    "NR": ANSWER_PROMPT_NR,
}

JUDGE_PROMPT_BY_TYPE = {
    "LR": JUDGE_PROMPT_LR,
    "PE": JUDGE_PROMPT_PE,
    "NR": JUDGE_PROMPT_NR,
}

In [5]:
import json
import re


def parse_judge(text):

    match = re.search(
        r"\{.*\}",
        text,
        flags=re.DOTALL
    )

    if not match:

        return {
            "score": None,
            "reason": text[:1000]
        }

    try:

        obj = json.loads(match.group(0))

        score = float(obj["score"])

        score = max(
            0.0,
            min(
                1.0,
                score
            )
        )

        return {
            "score": score,
            "reason": obj.get(
                "reason",
                ""
            )
        }

    except Exception:

        return {
            "score": None,
            "reason": text[:1000]
        }

In [6]:
def llm_call(prompt, temperature=0):
    messages = [{"role": "user", "content": prompt}]

    response, latency_ms = timed_completion(
        messages=messages,
        model=MODEL,
        n=MODEL_N,
        temperature=temperature,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )

    text             = strip_thinking(response.choices[0].message.content).strip()
    prompt_tokens    = response.usage.prompt_tokens
    completion_tokens = response.usage.completion_tokens

    pricing = COSTS.get(MODEL_N, {})
    if pricing.get("input") is not None:
        cost_usd = (
            prompt_tokens    * pricing["input"] +
            completion_tokens * pricing["output"]
        ) / 1_000_000
    else:
        cost_usd = None

    return text, latency_ms, prompt_tokens, completion_tokens, cost_usd

In [ ]:
VALID_QUESTION_TYPES = {"LR", "PE", "NR"}

def answer_and_judge(row):

    question      = str(row["Question"])
    raw_extracts  = row["Document extracts"]
    extracts      = str(raw_extracts) if pd.notna(raw_extracts) else ""
    ground_truth  = str(row["Answer"])
    question_type = str(row.get("Type of question", "")).strip().upper()

    if question_type not in VALID_QUESTION_TYPES:
        question_type = "LR"

    # ── Answer ───────────────────────────────────────────────────────────────
    answer_prompt = ANSWER_PROMPT_BY_TYPE[question_type].format(
        question=question,
        extracts=extracts,
    )
    model_answer, answer_latency_ms, ans_pt, ans_ct, ans_cost = llm_call(answer_prompt, temperature=0)

    # ── Judge ────────────────────────────────────────────────────────────────
    judge_prompt = JUDGE_PROMPT_BY_TYPE[question_type].format(
        question=question,
        extracts=extracts,
        ground_truth=ground_truth,
        model_answer=model_answer,
    )
    judge_text, judge_latency_ms, jdg_pt, jdg_ct, jdg_cost = llm_call(judge_prompt, temperature=0)
    judge = parse_judge(judge_text)

    return {
        "company":            row["Company's name"],
        "fiscal_year":        row["Fiscal year"],
        "question_id":        row["Question ID"],
        "question_type":      question_type,

        "perturbation_type":  row["perturbation_type"],
        "perturbation_level": row["perturbation_level"],

        "question":           question,
        "ground_truth":       ground_truth,
        "llm_answer":         model_answer,

        "answer_prompt_tokens":     ans_pt,
        "answer_completion_tokens": ans_ct,
        "answer_cost_usd":          ans_cost,
        "answer_latency_ms":        answer_latency_ms,

        "judge_score":              judge["score"],
        "judge_reason":             judge["reason"],
        "judge_raw_output":         judge_text,
        "judge_prompt_tokens":      jdg_pt,
        "judge_completion_tokens":  jdg_ct,
        "judge_cost_usd":           jdg_cost,
        "judge_latency_ms":         judge_latency_ms,
    }

In [ ]:
MAX_WORKERS = 28

results = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(answer_and_judge, row) for _, row in df_run.iterrows()]

    for future in tqdm(as_completed(futures), total=len(futures)):
        try:
            results.append(future.result())
        except Exception as e:
            print("\nERROR:", type(e), e)
            results.append({
                "company": None, "fiscal_year": None, "question_id": None,
                "question_type": None,
                "perturbation_type": None, "perturbation_level": None,
                "question": None, "ground_truth": None, "llm_answer": None,
                "answer_prompt_tokens": None, "answer_completion_tokens": None,
                "answer_cost_usd": None, "answer_latency_ms": None,
                "judge_score": None, "judge_reason": str(e), "judge_raw_output": None,
                "judge_prompt_tokens": None, "judge_completion_tokens": None,
                "judge_cost_usd": None, "judge_latency_ms": None,
            })

  0%|          | 13/6270 [00:32<4:21:36,  2.51s/it]


In [ ]:
results_df = pd.DataFrame(results)
results_df.head()

NameError: name 'pd' is not defined

In [ ]:
from datetime import datetime

OUTPUT_DIR = Path("../../datasets/experiments/llm/reasoning")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_FILE = OUTPUT_DIR / f"cfb_judged_{ts}.csv"

results_df.to_csv(OUTPUT_FILE, index=False)
print(OUTPUT_FILE.resolve())

/Users/anton/Documents/engd/courses/llmb/llmb-project-group-1/datasets/experiments/llm/classification/llm_reasoning_judged.csv
